In [ ]:
# Performance summary notebook for SpotWhisperer
import pandas as pd
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

print("Creating SpotWhisperer performance summary...")

In [ ]:
# Load all performance files
performance_data = []
for perf_file in snakemake.input.performance_files:
    with open(perf_file, "r") as f:
        data = json.load(f)
        performance_data.append(data)
        print(f'Loaded {perf_file}: {data["accuracy"]:.3f} accuracy')

df = pd.DataFrame(performance_data)
print(f"\nCombined performance data shape: {df.shape}")
print(df.head())

In [ ]:
# Create summary plot
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Accuracy plot
sns.barplot(data=df, x="dataset", y="accuracy", hue="metadata_col", ax=axes[0])
axes[0].set_title("Accuracy by Dataset and Metadata")
axes[0].set_ylim(0, 1)
axes[0].tick_params(axis="x", rotation=45)

# F1-score plot
sns.barplot(data=df, x="dataset", y="f1_weighted", hue="metadata_col", ax=axes[1])
axes[1].set_title("F1-Score by Dataset and Metadata")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()

# Save plot
Path(snakemake.output.plot).parent.mkdir(parents=True, exist_ok=True)
plt.savefig(snakemake.output.plot, dpi=300, bbox_inches="tight")
plt.close()

print(f"Saved performance summary plot to {snakemake.output.plot}")

In [ ]:
# Save summary CSV

df.to_csv(snakemake.output.summary, index=False)
print(f"Saved performance summary to {snakemake.output.summary}")

# Print summary statistics
print("\nSummary Statistics:")
print(f'Mean accuracy: {df["accuracy"].mean():.3f}')
print(f'Mean F1-score: {df["f1_weighted"].mean():.3f}')
print(
    f'Best performing: {df.loc[df["accuracy"].idxmax(), "dataset"]} - {df.loc[df["accuracy"].idxmax(), "metadata_col"]}'
)